<a href="https://colab.research.google.com/github/Anish-185/IPL-PRECTION-SYSTEM/blob/main/IPL_PREDICTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

In [1]:
import kagglehub
path = kagglehub.dataset_download("patrickb1912/ipl-complete-dataset-20082020")

100%|██████████| 1.82M/1.82M [00:00<00:00, 110MB/s]

Extracting files...


In [9]:
  # ================= IMPORTS =================
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import os

# Debugging: Print contents of the downloaded dataset directory to find the correct CSV path
print(f"Contents of the downloaded dataset directory ({path}):")
print(os.listdir(path))

# Load the dataset
# This line caused the FileNotFoundError. We will fix it after inspecting the output of os.listdir(path).
df = pd.read_csv(os.path.join(path, 'matches.csv'))

# Debugging: Print columns of the DataFrame to identify correct names
print("DataFrame columns:")
print(df.columns)

# 1. ENHANCED FEATURE ENGINEERING
def prepare_ipl_data(df):
    # Standardize names for 2026 Season
    team_map = {"Delhi Daredevils": "Delhi Capitals", "Kings XI Punjab": "Punjab Kings", "Royal Challengers Bangalore": "Royal Challengers Bengaluru"}
    df.replace(team_map, inplace=True)

    # Define 2026 Home Venues (including secondary grounds)
    home_venues = {
        "Chennai Super Kings": "MA Chidambaram Stadium",
        "Mumbai Indians": "Wankhede Stadium",
        "Royal Challengers Bengaluru": "M Chinnaswamy Stadium",
        "Kolkata Knight Riders": "Eden Gardens",
        "Delhi Capitals": "Arun Jaitley Stadium",
        "Rajasthan Royals": "Sawai Mansingh Stadium",
        "Sunrisers Hyderabad": "Rajiv Gandhi Intl Stadium",
        "Punjab Kings": "New PCA Stadium, Mullanpur",
        "Gujarat Titans": "Narendra Modi Stadium",
        "Lucknow Super Giants": "Ekana Cricket Stadium"
    }

    data_rows = []
    for _, row in df.iterrows():
        # Create two rows per match to make the model order-independent
        # Perspective 1: Team 1
        data_rows.append({
            'team': row['team1'], 'opponent': row['team2'], 'venue': row['venue'],
            'is_home': 1 if home_venues.get(row['team1']) == row['venue'] else 0,
            'result': 1 if row['winner'] == row['team1'] else 0
        })
        # Perspective 2: Team 2
        data_rows.append({
            'team': row['team2'], 'opponent': row['team1'], 'venue': row['venue'],
            'is_home': 1 if home_venues.get(row['team2']) == row['venue'] else 0,
            'result': 1 if row['winner'] == row['team2'] else 0
        })

    return pd.DataFrame(data_rows), home_venues

# 2. MODEL TRAINING
# Assuming 'df' is your historical dataset
processed_df, home_venues = prepare_ipl_data(df)
X = pd.get_dummies(processed_df.drop('result', axis=1))
y = processed_df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.05, eval_metric='logloss')
model.fit(X_train, y_train)

# 3. PREDICTION FUNCTIONS
def predict_winner(team1, team2, venue):
    # Create empty feature row based on training columns
    input_row = pd.DataFrame(0, index=[0], columns=X.columns)

    # Fill features
    if f'team_{team1}' in input_row: input_row[f'team_{team1}'] = 1
    if f'opponent_{team2}' in input_row: input_row[f'opponent_{team2}'] = 1
    if f'venue_{venue}' in input_row: input_row[f'venue_{venue}'] = 1
    input_row['is_home'] = 1 if home_venues.get(team1) == venue else 0

    prob = model.predict_proba(input_row)[0][1] # Probability Team 1 wins
    return team1 if prob > 0.5 else team2, prob

# 4. TOURNAMENT SIMULATOR
def simulate_2026_season():
    teams = list(home_venues.keys())
    standings = {t: 0 for t in teams}

    # Round Robin Simulation
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            # Each plays home and away
            for home_team, away_team in [(teams[i], teams[j]), (teams[j], teams[i])]:
                venue = home_venues[home_team]
                winner, _ = predict_winner(home_team, away_team, venue)
                standings[winner] += 2

    # Playoff Logic
    top4 = sorted(standings, key=standings.get, reverse=True)[:4]
    q1_win, _ = predict_winner(top4[0], top4[1], home_venues[top4[0]])
    elim_win, _ = predict_winner(top4[2], top4[3], home_venues[top4[2]])

    q1_loser = top4[1] if q1_win == top4[0] else top4[0]
    q2_win, _ = predict_winner(q1_loser, elim_win, home_venues[q1_loser])

    # Final (Neutral venue - usually Ahmedabad or Bengaluru)
    final_winner, _ = predict_winner(q1_win, q2_win, "Narendra Modi Stadium")
    return final_winner

# RUN


Contents of the downloaded dataset directory (/root/.cache/kagglehub/datasets/patrickb1912/ipl-complete-dataset-20082020/versions/3):
['deliveries.csv', 'matches.csv']
DataFrame columns:
Index(['id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2'],
      dtype='object')


In [10]:
home_venues_keys = list(home_venues.keys())
home_venues_values = list(home_venues.values())

print("Available Teams:")
for team in home_venues_keys:
    print(f"- {team}")

print("\nAvailable Venues:")
for venue in home_venues_values:
    print(f"- {venue}")

Available Teams:
- Chennai Super Kings
- Mumbai Indians
- Royal Challengers Bengaluru
- Kolkata Knight Riders
- Delhi Capitals
- Rajasthan Royals
- Sunrisers Hyderabad
- Punjab Kings
- Gujarat Titans
- Lucknow Super Giants

Available Venues:
- MA Chidambaram Stadium
- Wankhede Stadium
- M Chinnaswamy Stadium
- Eden Gardens
- Arun Jaitley Stadium
- Sawai Mansingh Stadium
- Rajiv Gandhi Intl Stadium
- New PCA Stadium, Mullanpur
- Narendra Modi Stadium
- Ekana Cricket Stadium


In [11]:
print("--- IPL 2026 Single Match Prediction ---")
t1 = input("Team 1: ")
t2 = input("Team 2: ")
ven = input("Venue: ")
winner, prob = predict_winner(t1, t2, ven)
print(f"Predicted Winner: {winner} ({prob*100:.1f}% confidence)")



--- IPL 2026 Single Match Prediction ---
Team 1: Lucknow Super Giants
Team 2: Chennai Super Kings
Venue: Ekana Cricket Stadium
Predicted Winner: Lucknow Super Giants (66.9% confidence)


In [12]:
print("\n--- Simulating 2026 Overall Winner ---")
season_winner = simulate_2026_season()
print(f"The predicted IPL 2026 Champion is: {season_winner} 🏆")


--- Simulating 2026 Overall Winner ---
The predicted IPL 2026 Champion is: Gujarat Titans 🏆
